# ASSIGNMENT 1 - Question 2 - Case Circuit Visión
### Edge Detection on a PCB Inspection Line
### Alumno: Martín Andrés Gutiérrez López
Topics 2D Convolution, stride, padding (valid / same)

In [5]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ==============================================================================
# CASE DATA: CircuitVision (PCB Inspection Line)
# ==============================================================================
# 5x5 input image patch
I = np.array(
    [
        [1, 2, 0, 1, 3],
        [0, 1, 2, 3, 1],
        [1, 0, 1, 2, 2],
        [2, 1, 0, 1, 0],
        [3, 2, 1, 0, 1],
    ],
    dtype=float,
)

# 3x3 filter/kernel (vertical edge detector)
K = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float)


# Generic 2D convolution function in NumPy
def conv2d_numpy(image, kernel, stride=1, padding=0):
  H, W = image.shape
  kH, kW = kernel.shape

  # Apply zero-padding if P > 0
  if padding > 0:
    padded = np.pad(
        image,
        ((padding, padding), (padding, padding)),
        mode="constant",
        constant_values=0,
    )
  else:
    padded = image

  pH, pW = padded.shape
  out_H = math.floor((pH - kH) / stride) + 1
  out_W = math.floor((pW - kW) / stride) + 1

  output = np.zeros((out_H, out_W))

  for r in range(out_H):
    for c in range(out_W):
      r_start = r * stride
      c_start = c * stride
      window = padded[r_start : r_start + kH, c_start : c_start + kW]
      output[r, c] = np.sum(window * kernel)

  return output


print(f"{BOLD_BLUE}=" * 60)
print(f"PART (a): Stride = 1, Valid Padding (P = 0)")
print("=" * 60,f"{RESET}")
out_a = conv2d_numpy(I, K, stride=1, padding=0)
print(f"Output shape: {out_a.shape}")
print("Resulting feature map:")
print(out_a.astype(int))

print(f"\n{BOLD_BLUE}")
print("=" * 60)
print("PART (b): Stride = 2, Valid Padding (P = 0)")
print("=" * 60,f"{RESET}")
out_b = conv2d_numpy(I, K, stride=2, padding=0)
print(f"Output shape: {out_b.shape}")
print("Resulting feature map:")
print(out_b.astype(int))
print("Skipped positions in full feature map: odd rows and odd columns.")

print(f"\n{BOLD_BLUE}")
print("=" * 60)
print("PART (c): Same Padding (P = 1, Stride = 1)")
print("=" * 60,f"{RESET}")
out_c = conv2d_numpy(I, K, stride=1, padding=1)
print(f"Output shape: {out_c.shape} (preserves 5x5)")
print(out_c.astype(int))
print(f"\n{BOLD}Top-left value [0, 0]{RESET}: {int(out_c[0, 0])}")

# ----------------------------------------------------------------
# Question 2d The production camera delivers 224 × 224 × 3 images. 
# The first layer is a 7 × 7 convolution with 64 filters, stride 2, padding 3, followed by 3 × 3 max pooling with stride 2, padding 1. 
# Using O = ⌊(N + 2P − K) / S⌋ + 1 give the output shape after each layer and the number of learnable parameters in the convolution

print(f"\n{BOLD_BLUE}")
print("=" * 60)
print("PART (d): ResNet-Style Pipeline (PyTorch Verification)")
print("=" * 60,f"{RESET}")


# PyTorch implementation to verify dimensions and parameter counts
class ResNetFirstStage(nn.Module):

  def __init__(self):
    super().__init__()
    # 7x7 Convolution, 64 filters, stride 2, padding 3
    self.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=7,
        stride=2,
        padding=3,
        bias=True,
    )
    # 3x3 Max pooling, stride 2, padding 1
    self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

  def forward(self, x):
    x_conv = self.conv1(x)
    x_pool = self.pool(x_conv)
    return x_conv, x_pool


model = ResNetFirstStage()

# Dummy input tensor with shape (Batch=1, Channels=3, Height=224, Width=224)
x_in = torch.randn(1, 3, 224, 224)
out_conv, out_pool = model(x_in)

# Count learnable parameters
weights_conv = model.conv1.weight.numel()
biases_conv = model.conv1.bias.numel()
total_params = weights_conv + biases_conv

print(f"Input shape                 : {list(x_in.shape)}")
print(f"Output shape post-conv      : {list(out_conv.shape)}  -> 112x112x64")
print(f"Output shape post-maxpool   : {list(out_pool.shape)}  -> 56x56x64")
print("-" * 60)
print(f"Filter weights (7x7x3x64)   : {weights_conv}")
print(f"Biases (64)                 : {biases_conv}")
print(f"Total learnable parameters  : {total_params} (9408 without bias)")

PART (a): Stride = 1, Valid Padding (P = 0)
Output shape: (3, 3)
Resulting feature map:
[[-1 -3 -3]
 [ 0 -4  0]
 [ 4  0 -1]]


PART (b): Stride = 2, Valid Padding (P = 0)
Output shape: (2, 2)
Resulting feature map:
[[-1 -3]
 [ 4 -1]]
Skipped positions in full feature map: odd rows and odd columns.


PART (c): Same Padding (P = 1, Stride = 1)
Output shape: (5, 5) (preserves 5x5)
[[-3 -1 -1 -2  4]
 [-3 -1 -3 -3  6]
 [-2  0 -4  0  6]
 [-3  4  0 -1  3]
 [-3  4  2  0  1]]

Top-left value [0, 0]: -3


PART (d): ResNet-Style Pipeline (PyTorch Verification)
Input shape                 : [1, 3, 224, 224]
Output shape post-conv      : [1, 64, 112, 112]  -> 112x112x64
Output shape post-maxpool   : [1, 64, 56, 56]  -> 56x56x64
------------------------------------------------------------
Filter weights (7x7x3x64)   : 9408
Biases (64)                 : 64
Total learnable parameters  : 9472 (9408 without bias)
